In [1]:
from pathlib import Path
import pandas as pd

txt_path = Path("one_month.txt")  # same folder as notebook

# 1) Read each line as a single string
raw = pd.read_csv(txt_path, sep="\t", header=None, names=["line"], engine="python")

# 2) Optional: parse log fields into columns
df = raw["line"].str.extract(
    r"^(?P<date>\d{4}-\d{2}-\d{2})\s+"
    r"(?P<time>\d{2}:\d{2}:\d{2},\d{3})\s+"
    r"(?P<logger>\S+)\s+"
    r"(?P<level>\S+)\s+"
    r"(?P<message>.*)$"
)

display(df.head())

,date,time,logger,level,message
0,2026-03-18,"16:28:04,693",pcrglobwb,INFO,Updating model for time 1950-05-01
1,2026-03-18,"16:28:04,694",virtualOS,DEBUG,reading variable: precipitation from the file:...
2,2026-03-18,"16:28:04,694",virtualOS,DEBUG,Finding the date based on the given climatolog...
3,2026-03-18,"16:28:04,694",virtualOS,DEBUG,Using the date index 4
4,2026-03-18,"16:28:04,694",virtualOS,DEBUG,Using the datetime 1961-05-01 00:00:00


In [2]:
# Build one timestamp column from date + time
df["timestamp"] = pd.to_datetime(
    df["date"] + " " + df["time"],
    format="%Y-%m-%d %H:%M:%S,%f",
    errors="coerce",
)

# Optional: keep both for checking
df["duration_from_prev_ms"] = df["timestamp"].diff().dt.total_seconds() * 1000
df["duration_to_next_ms"] = (df["timestamp"].shift(-1) - df["timestamp"]).dt.total_seconds() * 1000

# Use this for "process duration per message"
df["duration_ms"] = df["duration_from_prev_ms"].fillna(0)


# Duration in milliseconds
#df["duration_ms"] = df["duration"].dt.total_seconds() * 1000

display(df.head(10))

,date,time,logger,level,message,timestamp,duration_from_prev_ms,duration_to_next_ms,duration_ms
0,2026-03-18,"16:28:04,693",pcrglobwb,INFO,Updating model for time 1950-05-01,2026-03-18 16:28:04.693,NaN,1.0,0.0
1,2026-03-18,"16:28:04,694",virtualOS,DEBUG,reading variable: precipitation from the file:...,2026-03-18 16:28:04.694,1.0,0.0,1.0
2,2026-03-18,"16:28:04,694",virtualOS,DEBUG,Finding the date based on the given climatolog...,2026-03-18 16:28:04.694,0.0,0.0,0.0
3,2026-03-18,"16:28:04,694",virtualOS,DEBUG,Using the date index 4,2026-03-18 16:28:04.694,0.0,0.0,0.0
4,2026-03-18,"16:28:04,694",virtualOS,DEBUG,Using the datetime 1961-05-01 00:00:00,2026-03-18 16:28:04.694,0.0,76.0,0.0
5,2026-03-18,"16:28:04,770",virtualOS,DEBUG,sameClone=False,2026-03-18 16:28:04.770,76.0,1.0,76.0
6,2026-03-18,"16:28:04,771",virtualOS,DEBUG,"cellsizeInput=0.5, cellsizeClone=0.08333333333...",2026-03-18 16:28:04.771,1.0,0.0,1.0
7,2026-03-18,"16:28:04,771",virtualOS,DEBUG,"rowsInput=360, rowsClone=180.0",2026-03-18 16:28:04.771,0.0,0.0,0.0
8,2026-03-18,"16:28:04,771",virtualOS,DEBUG,"colsInput=720, colsClone=288.0",2026-03-18 16:28:04.771,0.0,0.0,0.0
9,2026-03-18,"16:28:04,771",virtualOS,DEBUG,"xULInput=-180.0, xULClone=53.0",2026-03-18 16:28:04.771,0.0,0.0,0.0


In [3]:
df_sorted = df.sort_values(by="duration_ms", ascending=False, na_position="last").reset_index(drop=True)
display(df_sorted[['date', 'time', 'logger', 'level', 'message','duration_ms']].head(20))

,date,time,logger,level,message,duration_ms
0,2026-03-18,"16:28:56,102",virtualOS,DEBUG,Input map and clone map are different. Croppin...,1594.0
1,2026-03-18,"16:29:13,252",virtualOS,DEBUG,Input map and clone map are different. Croppin...,1281.0
2,2026-03-18,"16:29:20,257",virtualOS,DEBUG,Input map and clone map are different. Croppin...,1268.0
3,2026-03-18,"16:28:48,788",virtualOS,DEBUG,Input map and clone map are different. Croppin...,1197.0
4,2026-03-18,"16:29:16,411",virtualOS,DEBUG,Input map and clone map are different. Croppin...,1191.0
5,2026-03-18,"16:28:06,835",virtualOS,DEBUG,Input map and clone map are different. Croppin...,1186.0
6,2026-03-18,"16:29:09,731",virtualOS,DEBUG,Input map and clone map are different. Croppin...,1178.0
7,2026-03-18,"16:29:38,129",virtualOS,DEBUG,Input map and clone map are different. Croppin...,1150.0
8,2026-03-18,"16:28:11,261",virtualOS,DEBUG,Input map and clone map are different. Croppin...,1057.0
9,2026-03-18,"16:28:52,524",virtualOS,DEBUG,Input map and clone map are different. Croppin...,1009.0


In [4]:
pd.set_option("display.max_colwidth", None)

df_sorted = df.sort_values(by="duration_ms", ascending=False, na_position="last").reset_index(drop=True)
view = df_sorted[["date", "time", "logger", "level", "message", "duration_ms"]].head(40)

display(
    view.style.set_properties(
        subset=["message"],
        **{
            "max-width": "1200px",
            "white-space": "pre-wrap",
        },
    )
)

,date,time,logger,level,message,duration_ms
0,2026-03-18,"16:28:56,102",virtualOS,DEBUG,Input map and clone map are different. Cropping/resampling is needed.,1594.000000
1,2026-03-18,"16:29:13,252",virtualOS,DEBUG,Input map and clone map are different. Cropping/resampling is needed.,1281.000000
2,2026-03-18,"16:29:20,257",virtualOS,DEBUG,Input map and clone map are different. Cropping/resampling is needed.,1268.000000
3,2026-03-18,"16:28:48,788",virtualOS,DEBUG,Input map and clone map are different. Cropping/resampling is needed.,1197.000000
4,2026-03-18,"16:29:16,411",virtualOS,DEBUG,Input map and clone map are different. Cropping/resampling is needed.,1191.000000
5,2026-03-18,"16:28:06,835",virtualOS,DEBUG,Input map and clone map are different. Cropping/resampling is needed.,1186.000000
6,2026-03-18,"16:29:09,731",virtualOS,DEBUG,Input map and clone map are different. Cropping/resampling is needed.,1178.000000
7,2026-03-18,"16:29:38,129",virtualOS,DEBUG,Input map and clone map are different. Cropping/resampling is needed.,1150.000000
8,2026-03-18,"16:28:11,261",virtualOS,DEBUG,Input map and clone map are different. Cropping/resampling is needed.,1057.000000
9,2026-03-18,"16:28:52,524",virtualOS,DEBUG,Input map and clone map are different. Cropping/resampling is needed.,1009.000000


In [5]:


# Group by message and sum runtime
df["duration_ms"] = pd.to_numeric(df["duration_ms"], errors="coerce")

df["process"] = (
    df["message"]
      .str.replace(r"\b\d{4}-\d{2}-\d{2}\b", "<DATE>", regex=True)
      .str.replace(r"\b\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}\b", "<DATETIME>", regex=True)
)

total_runtime_ms = df["duration_ms"].sum(skipna=True)

msg_runtime = (
    df.groupby("process", dropna=False, as_index=False)
      .agg(total_runtime_ms=("duration_ms", "sum"), calls=("process", "size"))
      .sort_values("total_runtime_ms", ascending=False)
      .reset_index(drop=True)
)

msg_runtime["fraction_of_total"] = msg_runtime["total_runtime_ms"] / total_runtime_ms if total_runtime_ms else 0.0
msg_runtime["percent_of_total"] = msg_runtime["fraction_of_total"] * 100

display(msg_runtime.head(20))



,process,total_runtime_ms,calls,fraction_of_total,percent_of_total
0,sameClone=False,43252.0,442,0.355638,35.563815
1,Input map and clone map are different. Cropping/resampling is needed.,43043.0,442,0.353920,35.391965
2,reporting for time <DATE>,4792.0,31,0.039402,3.940206
3,Allocation of supply from desalination water.,4589.0,124,0.037733,3.773290
4,Updating groundwater,2849.0,31,0.023426,2.342581
5,Total groundwater abstraction is limited by regional annual pumping capacity.,2607.0,155,0.021436,2.143597
6,Fossil groundwater abstractions are allowed.,2489.0,124,0.020466,2.046572
7,Allocation of surface water abstraction.,2166.0,124,0.017810,1.780986
8,Updating land cover: grassland,1842.0,31,0.015146,1.514578
9,Updating land cover: irrPaddy,1793.0,31,0.014743,1.474288


In [6]:
reordered_msg_runtime = msg_runtime[["total_runtime_ms", "percent_of_total", "process"]].copy()
reordered_msg_runtime["total_runtime_ms"] = reordered_msg_runtime["total_runtime_ms"].astype(int)


styled = (
    reordered_msg_runtime.head(20).style
    .format({"percent_of_total": "{:.2f}"}, escape="html")  # keeps <DATE> visible
    .set_properties(
        subset=["process"],
        **{
            "text-align": "left",
            "max-width": "1000px",
            "white-space": "pre-wrap",
        },
    )
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
)


display(styled)


,total_runtime_ms,percent_of_total,process
0,43252,35.56,sameClone=False
1,43043,35.39,Input map and clone map are different. Cropping/resampling is needed.
2,4792,3.94,reporting for time <DATE>
3,4589,3.77,Allocation of supply from desalination water.
4,2849,2.34,Updating groundwater
5,2607,2.14,Total groundwater abstraction is limited by regional annual pumping capacity.
6,2489,2.05,Fossil groundwater abstractions are allowed.
7,2166,1.78,Allocation of surface water abstraction.
8,1842,1.51,Updating land cover: grassland
9,1793,1.47,Updating land cover: irrPaddy
